# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft>=0.19.0 bitsandbytes accelerate datasets pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# added PEFT
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm
import shutil

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/content/drive/MyDrive/m1/7123 DL/pixels-to-predictions")
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 384

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Preprocess Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
  df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row, include_answer=False):
  parts = []
  for field in ["lecture", "hint"]:
    val = row.get(field, "")
    if pd.notna(val) and str(val).strip():
        parts.append(str(val).strip())
  context = "\n".join(parts)
  choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))

  prompt = "<image>\n"
  prompt += f"Question: {row['question']}\n"

  subject = row.get("subject", "")
  if pd.notna(subject) and str(subject).strip():
    prompt += f"Subject: {subject}\n"

  if context:
    prompt += f"Supporting Information:\n{context}\n\n"
  prompt += f"Choices:\n{choices_str}\n"
  prompt += "The correct answer is:"

  if include_answer:
    prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"
  return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
  def __init__(self, df, data_dir, img_size=384, is_train=True):
    self.df = df.reset_index(drop=True)
    self.data_dir = data_dir
    self.img_size = img_size
    self.is_train = is_train

  def __len__(self):
    return len(self.df)

  def _load_image(self, rel_path):
    expected_path = self.data_dir / "images" / "images" / rel_path.replace("images/", "", 1)
    if not expected_path.exists():
      fallback = self.data_dir / rel_path
      if fallback.exists():
        expected_path = fallback
      else:
        raise FileNotFoundError(f"Image not found: {rel_path}")
    img = Image.open(expected_path).convert("RGB")
    return img.resize((self.img_size, self.img_size), Image.BICUBIC)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img = self._load_image(row["image_path"])
    base = {"image": img, "text": build_prompt(row, include_answer=self.is_train), "id": row["id"]}
    if self.is_train:
      base["answer"] = int(row["answer"])
    else:
      base["choices"] = row["choices"]
      base["answer"] = int(row.get("answer", -1))
    return base

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
def collate_fn(batch):
  images = [item["image"] for item in batch]
  texts  = [item["text"] for item in batch]

  inputs = processor(
      text=texts, images=images,
      return_tensors="pt", padding=True,
  )

  return {
      "pixel_values": inputs["pixel_values"],
      "input_ids": inputs["input_ids"],
      "attention_mask": inputs["attention_mask"],
      "labels": inputs["input_ids"].clone(),
      "answers": torch.tensor([item["answer"] for item in batch]),
      "ids": [item["id"] for item in batch],
  }

In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
BATCH_SIZE = 4
MAX_TRAINABLE_PARAMS = 5_000_000

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.size = {"longest_edge": IMG_SIZE}
if processor.tokenizer.pad_token is None:
  processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
)
model = prepare_model_for_kbit_training(model)

# added training prep + lora configs
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=6,       # 8->6
    lora_alpha=24,  # 32->24
    use_dora=True,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,} / {MAX_TRAINABLE_PARAMS:,}")

In [ ]:
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=2e-4)
dataloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

EPOCHS = 3
peft_model.train()
for epoch in range(EPOCHS):
  total_loss = 0
  pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
  for batch in pbar:
    batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
    loss = peft_model(**batch).loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})
  print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(dataloader):.4f}")

SAVE_PATH = "/content/drive/MyDrive/m1/7123 DL/lora_model_v11"
peft_model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

In [ ]:
@torch.no_grad()
def predict_test(model, dataset):
  model.eval()
  preds = {}
  for idx in tqdm(range(len(dataset)), desc="Predicting"):
    sample = dataset[idx]
    row = dataset.df.iloc[idx]
    choices = row["choices"]
    losses = []
    for ci in range(len(choices)):
      prompt = build_prompt(row, include_answer=False) + f" {CHOICE_LETTERS[ci]}"
      inp = processor(text=[prompt], images=[sample["image"]], return_tensors="pt")
      inp = {k: v.to(model.device) for k, v in inp.items()}
      inp["labels"] = inp["input_ids"].clone()
      outputs = model(**inp)
      losses.append(outputs.loss.item())
    preds[row["id"]] = int(np.argmin(losses))
  return preds

test_preds = predict_test(peft_model, test_ds)
submission = pd.DataFrame({"id": list(test_preds.keys()), "answer": list(test_preds.values())})
submission.to_csv("/content/drive/MyDrive/m1/7123 DL/submission_v11.csv", index=False)